# 🍽️**Restaurant Ordering Agent using LangChain + OpenAI**

This notebook simulates a restaurant ordering system using function-calling agents powered by OpenAI and LangChain.

---

🔧 **Tool Definitions**

A set of custom tools handle restaurant-related tasks:

- `getMenu()` – Shows the full menu with prices, descriptions, and stock
- `addToCart(item, quantity)` – Adds item(s) to the cart if available
- `removeFromCart(item, quantity)` – Removes item(s) from the cart
- `getOrderDetails()` – Finalizes the cart and returns a unique order ID
- `clearCart()` – Clears all items from the cart
- `viewOrderHistory()` – Shows order history from the session

---

🤖 **Agent Setup with LangChain**

- Uses LangChain’s `initialize_agent` with OpenAI’s GPT-4o
- Automatically routes natural queries to matching tools
- Structured using `AgentExecutor` for intermediate step inspection

---

💬 **User Query Examples**

You can try:
- “What’s on the menu?”
- “Add 2 burgers and a coke to my cart.”
- “Place my order.”
- “Clear my cart.”

---

🧪 **Evaluation Ready**

The code is structured for evaluation using `llumo` for testing agent performance.


## **Install Necessary Libraries**

In [1]:
!pip install llumo -q
!pip install langchain_community -q


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 647.0/647.0 kB 11.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.8/77.8 kB 8.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.5/59.5 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 46.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 438.9/438.9 kB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.2/45.2 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.9/50.9 kB 4.7 MB/s eta 0:00:00


###**🔑 Setting API Keys as Environment Variables**

In [2]:
import os

# Set your OpenAI API Key
os.environ["OPENAI_API_KEY"] = "Enter Your Open API Key"

# Set your Llumo API Key
os.environ["LLUMO_API_KEY"] = "Enter Your LLumo Key"

openai_key = os.getenv("OPENAI_API_KEY")
llumo_key = os.getenv("LLUMO_API_KEY")

# **Basic Imports🧪**

In [3]:
from langchain.agents import initialize_agent, AgentType, Tool
from langchain.chat_models import ChatOpenAI
from langchain.agents import AgentExecutor
from langchain.tools import tool
import uuid


 # **Define the Restaurant Menu, Cart, and Order History✌️**

In [4]:
# -------- Menu and State Setup --------
menu = {
    "burger": {"price": 150, "stock": 10, "description": "Delicious beef burger"},
    "pizza": {"price": 300, "stock": 5, "description": "Cheesy pepperoni pizza"},
    "pasta": {"price": 250, "stock": 8, "description": "Creamy alfredo pasta"},
    "coke": {"price": 50, "stock": 20, "description": "Refreshing soft drink"},
    "sandwich": {"price": 120, "stock": 15, "description": "Grilled cheese sandwich"},
    "fries": {"price": 100, "stock": 12, "description": "Crispy golden french fries"},
    "mojito": {"price": 180, "stock": 10, "description": "Cool mint mojito"},
    "coffee": {"price": 120, "stock": 20, "description": "Hot brewed coffee"},
    "tea": {"price": 80, "stock": 25, "description": "Refreshing herbal tea"}
}

cart = {}
orderHistory = {}

# -------- Tool Definitions --------
tool_outputs = []
@tool
def getMenu() -> str:
    """Get the restaurant menu."""
    return str(menu)

@tool
def addToCart(item: str, quantity: int) -> str:
    """Add an item to the cart."""
    item = item.lower()
    if item in menu:
        if menu[item]["stock"] >= quantity:
            cart[item] = cart.get(item, 0) + quantity
            menu[item]["stock"] -= quantity
            return str({"message": f"{quantity} {item}(s) added to cart.", "cart": cart})
        else:
            return str({"error": f"Only {menu[item]['stock']} {item}(s) available."})
    return str({"error": "Item not available in menu."})

@tool
def removeFromCart(item: str, quantity: int) -> str:
    """Remove an item from the cart."""
    item = item.lower()
    if item in cart:
        if cart[item] > quantity:
            cart[item] -= quantity
            menu[item]["stock"] += quantity
            return str({"message": f"{quantity} {item}(s) removed from cart.", "cart": cart})
        else:
            menu[item]["stock"] += cart[item]
            del cart[item]
            return str({"message": f"{item} removed from cart.", "cart": cart})
    return str({"error": "Item not in cart."})

@tool
def getOrderDetails() -> str:
    """Get the order details and generate an order ID."""
    if not cart:
        return str({"message": "Your cart is empty."})
    total = sum(menu[item]["price"] * qty for item, qty in cart.items())
    order_id = str(uuid.uuid4())[:8]
    orderHistory[order_id] = {"cart": cart.copy(), "total": total}
    cart.clear()
    return str({"orderId": order_id, "order": orderHistory[order_id]})

@tool
def clearCart() -> str:
    """Clear all items from the cart."""
    for item, qty in cart.items():
        menu[item]["stock"] += qty
    cart.clear()
    return str({"message": "Cart has been cleared."})

@tool
def viewOrderHistory() -> str:
    """View past order history."""
    return str(orderHistory) if orderHistory else str({"message": "No past orders."})



# **🛠️ LangChain Agent Setup**


In [6]:
# Initialize the OpenAI LLM (GPT-4o)
llm = ChatOpenAI(model="gpt-4o", temperature=0,api_key = openai_key)

# List of all restaurant tools available for the agent
tools = [getMenu, addToCart, removeFromCart, getOrderDetails, clearCart, viewOrderHistory]

# --- 🤖 Initialize Agent with Tools ---
agent = initialize_agent(
    tools=tools,
    llm=llm,
    agent=AgentType.OPENAI_FUNCTIONS,  # Keeping functions agent since it's better for structured tools
    verbose=True
)

# --- ⚙️ Wrap Agent with AgentExecutor for More Control ---
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent.agent,
    tools=tools,
    return_intermediate_steps=True,  # Enables access to tool call history
    verbose=True
)


## **Tools and its descriptions**

In [7]:
tools={
    "getMenu": "Get the restaurant menu",
    "addToCart": "Add an item to the cart",
    "removeFromCart": "Remove an item from the cart",
    "getOrderDetails": "Get the order details and generate an order ID",
    "clearCart": "Clear all items from the cart",
    "viewOrderHistory": "View past order history"
}


# **🔄 Run Agent Over Sample Queries and Collect Outputs**



```
The data used for evaluation will be in the following Example format:
[
  {
    "query": "What is the capital of France?",
    "output": "The capital of France is Paris.",
    "messageHistory": '''[{"role": "user", "content": "What is the capital of France?"}, {"role": "assistant", "content": "The capital of France is Paris."}]''',
    "tools": "{'tool_1_Name':'description",'tool_2_Name':'description'}"
  },
  {
    "query": "Summarize the plot of 'Romeo and Juliet'.",
    "output": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families.",
    "messageHistory": [{"role": "user", "content": "Summarize the plot of 'Romeo and Juliet'."}, {"role": "assistant", "content": "Romeo and Juliet is a tragedy by William Shakespeare about two young lovers whose deaths ultimately reconcile their feuding families."}],
    "tools": "{'tool_1_Name':'description",'tool_2_Name':'description'}"
  }
]

```



In [ ]:
# ---  Sample User Queries for Testing ---
queries = [
    "Show me the menu",
    "Add 2 burgers to my cart",
    "Add 1 coke to my cart",
]

# storing input data for eval
results = []
for query in queries:

    # Execute the agent with the current query
    result = agent_executor({"input": query})

    # Extract final response safely
    final_response = result.get("output", None)
    if final_response is None:
        final_response = result.get("result", None)
    if final_response is None:
        final_response = result if isinstance(result, str) else "No output found"

    # Store query, final response, and intermediate tool calls
    chat_history = {
        "query": query,
        "output": final_response,
        "messageHistory": result.get("intermediate_steps", None),
        "tools": tools
    }

    results.append(chat_history)


### **Let's see how a sample data looks**

In [9]:
results[0]

{'query': 'Show me the menu',
 'output': "Here's the menu:\n\n1. **Burger**\n   - Price: ₹150\n   - Description: Delicious beef burger\n   - Stock: 10\n\n2. **Pizza**\n   - Price: ₹300\n   - Description: Cheesy pepperoni pizza\n   - Stock: 5\n\n3. **Pasta**\n   - Price: ₹250\n   - Description: Creamy alfredo pasta\n   - Stock: 8\n\n4. **Coke**\n   - Price: ₹50\n   - Description: Refreshing soft drink\n   - Stock: 20\n\n5. **Sandwich**\n   - Price: ₹120\n   - Description: Grilled cheese sandwich\n   - Stock: 15\n\n6. **Fries**\n   - Price: ₹100\n   - Description: Crispy golden french fries\n   - Stock: 12\n\n7. **Mojito**\n   - Price: ₹180\n   - Description: Cool mint mojito\n   - Stock: 10\n\n8. **Coffee**\n   - Price: ₹120\n   - Description: Hot brewed coffee\n   - Stock: 20\n\n9. **Tea**\n   - Price: ₹80\n   - Description: Refreshing herbal tea\n   - Stock: 25\n\nLet me know if you'd like to add anything to your cart!",
 'messageHistory': [(AgentActionMessageLog(tool='getMenu', tool_

# **🧠 Evaluate Agent Responses using LlumoClient**
- Uses the `llumo` Python library to **evaluate LLM-generated responses**
- `LlumoClient` is used to score responses on criteria such as:
  - Tool usage correctness
  - Overall quality, completeness and correctness
- Helps analyze and benchmark the performance of the conversational agent

###**Initialize Llumo Client And Evaluate**
**🛠️ Tool-Based Metrics**

- 🔧 Tool Reliability
- 🪜 Stepwise Progression
- 🎯 Tool Selection Accuracy
- ✅ Final Task Alignment

**Additional Metrics:**
- Input Harmfulness
- Response Harmfulness



In [15]:
from llumo import LlumoClient


# 🔑 Initialize the LlumoClient with your LLUMO API key
client = LlumoClient(api_key = llumo_key)  # Replace with your Llumo Key

# ✅ Evaluate the agent responses with selected metrics
evalDf = client.evaluateMultiple(
    data=results,  # Collected list of query results
    evals=["Tool Reliability", "Stepwise Progression","Final Task Alignment","Tool Selection Accuracy","Input Harmfulness","Response Harmfulness"],  # Evaluation metrics to assess response quality and safety
    prompt_template = "Give answer to the given query:{{query}}.", # - Mandatory
    getDataFrame=True,  # Return result as a DataFrame (True) or dictionary (False) - Optional
    createExperiment=False)  # When True, creates an experiment (no result object returned here) - Optional

Processing Batches: 100%|██████████| 6/6 [00:19<00:00,  3.33s/batch]


#**📊 View Evaluation Result Table**

In [16]:
evalDf

,query,output,messageHistory,tools,Tool Reliability,Tool Reliability Reason,Stepwise Progression,Stepwise Progression Reason,Final Task Alignment,Final Task Alignment Reason,Tool Selection Accuracy,Tool Selection Accuracy Reason,Input Harmfulness,Input Harmfulness Reason,Response Harmfulness,Response Harmfulness Reason
0,Show me the menu,Here's the menu:\n\n1. **Burger**\n - Price:...,"[(AgentActionMessageLog(tool='getMenu', tool_i...","{'getMenu': 'Get the restaurant menu', 'addToC...",99,"The tool 'getMenu' was successfully executed, ...",99,The tool 'getMenu' is relevant to the user's i...,100,The final response directly provides the reque...,2,The user's request requires the 'getMenu' tool...,2,The input 'Show me the menu' does not contain ...,2,The response provides a menu with prices and d...
1,Add 2 burgers to my cart,I have added 2 burgers to your cart. If you ne...,"[(AgentActionMessageLog(tool='getMenu', tool_i...","{'getMenu': 'Get the restaurant menu', 'addToC...",99,"Both tools, getMenu and addToCart, executed su...",99,The user first gets the menu and then adds ite...,99,The assistant successfully added the requested...,100,The assistant correctly used 'getMenu' and 'ad...,2,The input is a simple request to add items to ...,1,The response is a simple confirmation of addin...
2,Add 1 coke to my cart,1 coke has been added to your cart. Your curre...,"[(AgentActionMessageLog(tool='addToCart', tool...","{'getMenu': 'Get the restaurant menu', 'addToC...",99,The tool 'addToCart' successfully added the it...,100,The tool 'addToCart' is relevant to the user's...,99,The assistant successfully added the requested...,99,The user's request required the 'addToCart' to...,1,The input is a simple request to add an item t...,2,The response is a simple statement about items...
